# ChestAI — Training Notebook (Kaggle P100)

Run this notebook on **Kaggle** with the NIH Chest X-rays dataset attached.

**Setup:** Kernel → GPU P100 ON · Internet ON · Persistence ON

Dataset to attach: `nih-chest-xrays` (already on Kaggle, search in Add Data)

In [ ]:
# ── 0. Install dependencies ──────────────────────────────────────────────────
!pip install -q open_clip_torch timm wandb huggingface_hub groq slowapi
!pip install -q scikit-learn pandas numpy matplotlib seaborn opencv-python-headless
import subprocess, sys
print('Dependencies installed.')

In [ ]:
# ── 1. Clone project repo ────────────────────────────────────────────────────
!git clone https://github.com/YOUR_USERNAME/chestai.git /kaggle/working/chestai
%cd /kaggle/working/chestai
import sys; sys.path.insert(0, '/kaggle/working/chestai')
print('Project loaded.')

In [ ]:
# ── 2. W&B login (get key from wandb.ai/authorize) ───────────────────────────
import wandb
wandb.login(key='YOUR_WANDB_API_KEY')  # replace with your key
print('W&B logged in.')

In [ ]:
# ── 3. Verify GPU ────────────────────────────────────────────────────────────
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'CUDA: {torch.version.cuda}')

In [ ]:
# ── 4. Load config ───────────────────────────────────────────────────────────
import yaml
with open('configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

# Verify dataset paths
import os
print('images dir:', os.path.exists(cfg['data']['root_dir']))
print('labels csv:', os.path.exists(cfg['data']['labels_csv']))
print('train list:', os.path.exists(cfg['data']['train_val_list']))
print('test list: ', os.path.exists(cfg['data']['test_list']))

In [ ]:
# ── 5. EDA: Dataset statistics ───────────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from data.dataset import ChestXrayDataset, build_transforms

df = pd.read_csv(cfg['data']['labels_csv'])
print(f'Total images: {len(df):,}')
print(f'Unique patients: {df["Patient ID"].nunique():,}')
print(f'Age range: {df["Patient Age"].min()} – {df["Patient Age"].max()}')
print(f'Gender split: {df["Patient Gender"].value_counts().to_dict()}')

# Class prevalence
tf = build_transforms('val', 224)
full_ds = ChestXrayDataset(cfg['data']['root_dir'], cfg['data']['labels_csv'],
                            cfg['data']['train_val_list'], transform=tf)
stats = full_ds.label_stats()
print('\nClass prevalence:')
print(stats.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 4))
sns.barplot(data=stats, x='class', y='prevalence', ax=ax, palette='Blues_r')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.set_title('NIH ChestX-ray14 — Class Prevalence')
ax.set_ylabel('Proportion of samples')
plt.tight_layout()
plt.savefig('eda_prevalence.png', dpi=150)
plt.show()

In [ ]:
# ── 6. Train ─────────────────────────────────────────────────────────────────
from training.trainer import Trainer

# Optionally override config values for this run:
cfg['project']['entity'] = 'YOUR_WANDB_USERNAME'
cfg['training']['epochs'] = 30
cfg['training']['batch_size'] = 32

trainer = Trainer(cfg)
trainer.fit()
print(f'\nBest validation AUC: {trainer.best_auc:.4f}')

In [ ]:
# ── 7. Evaluate on held-out test set ─────────────────────────────────────────
import numpy as np
import torch
from torch.utils.data import DataLoader
from data.dataset import build_datasets
from data.transforms import build_transforms
from models.classifier import ChestAIClassifier
from models.uncertainty import mc_predict
from training.metrics import compute_metrics, metrics_dataframe
from fairness.audit import FairnessAuditor

device = torch.device('cuda')
datasets = build_datasets(cfg)
test_loader = DataLoader(datasets['test'], batch_size=64, num_workers=4)

# Load best checkpoint
ckpt = torch.load(f"{cfg['training']['checkpoint_dir']}/best_model.pt", map_location=device)
model = ChestAIClassifier()
model.load_state_dict(ckpt['model_state_dict'])
model.to(device).eval()

all_targets, all_probs, all_std, all_meta = [], [], [], []
for batch in test_loader:
    imgs, labels, meta = batch
    imgs = imgs.to(device)
    with torch.no_grad():
        mc = mc_predict(model, imgs, n_samples=20)
    all_probs.append(mc['mean'].cpu().numpy())
    all_std.append(mc['std'].cpu().numpy())
    all_targets.append(labels.numpy())
    all_meta.extend([{'age': a.item(), 'gender': g} for a, g in zip(meta['age'], meta['gender'])])

targets = np.concatenate(all_targets)
probs   = np.concatenate(all_probs)

test_metrics = compute_metrics(targets, probs, prefix='test/')
df_metrics = metrics_dataframe(test_metrics)
print('\n=== Test Set Results ===')
print(df_metrics.to_string(index=False))
print(f'\nMacro AUC: {test_metrics["test/macro/auc"]:.4f}')
print(f'Macro Sensitivity: {test_metrics["test/macro/sensitivity"]:.4f}')
print(f'Macro Specificity: {test_metrics["test/macro/specificity"]:.4f}')

In [ ]:
# ── 8. Fairness audit ────────────────────────────────────────────────────────
from fairness.audit import FairnessAuditor

auditor = FairnessAuditor(cfg)
report = auditor.run(targets, probs, all_meta)
auditor.save_report(report, 'fairness_report.json')

print('\n=== Fairness Audit ===')
print(f'Disparities flagged: {report["summary"]["total_disparities_flagged"]}')
if report['summary']['worst_disparity']:
    wd = report['summary']['worst_disparity']
    print(f'Worst: {wd["class"]} in {wd["subgroup"]} — ΔAUC={wd["delta"]:.3f}')
print(auditor.to_dataframe(report).to_string())

In [ ]:
# ── 9. GradCAM visualization ─────────────────────────────────────────────────
import matplotlib.pyplot as plt
from PIL import Image
from data.transforms import build_transforms
from explainability.gradcam import ViTGradCAM
from data.dataset import CLASSES

gradcam = ViTGradCAM(model)
tf = build_transforms('val', 224)

# Pick a random test image with a confirmed finding
test_ds = datasets['test']
for idx in range(len(test_ds)):
    img, label, meta = test_ds[idx]
    if label.sum() > 0:
        break

pil_img = Image.open(test_ds.root_dir / test_ds.df.iloc[idx]['Image Index']).convert('RGB')
tensor = tf(pil_img).unsqueeze(0).to(device)

fig, axes = plt.subplots(1, min(4, int(label.sum().item())) + 1, figsize=(20, 4))
axes[0].imshow(pil_img, cmap='gray'); axes[0].set_title('Original'); axes[0].axis('off')

true_classes = [CLASSES[i] for i in range(14) if label[i] > 0]
for i, cls in enumerate(true_classes[:4]):
    cls_idx = CLASSES.index(cls)
    heatmap = gradcam.generate(tensor, cls_idx)
    overlay = gradcam.overlay(pil_img, heatmap)
    axes[i+1].imshow(overlay); axes[i+1].set_title(cls); axes[i+1].axis('off')

plt.suptitle('GradCAM Heatmaps — Positive Classes')
plt.tight_layout()
plt.savefig('gradcam_examples.png', dpi=150, bbox_inches='tight')
plt.show()
print('GradCAM saved.')

In [ ]:
# ── 10. Export model to HuggingFace Hub ──────────────────────────────────────
from huggingface_hub import HfApi, login
import shutil

HF_TOKEN = 'YOUR_HF_TOKEN'  # get from huggingface.co/settings/tokens
HF_REPO  = 'YOUR_HF_USERNAME/chestai-model'

login(token=HF_TOKEN)
api = HfApi()

# Create repo if it doesn't exist
api.create_repo(repo_id=HF_REPO, exist_ok=True, private=False)

# Copy best checkpoint
best_path = f"{cfg['training']['checkpoint_dir']}/best_model.pt"
shutil.copy(best_path, '/kaggle/working/chestai_best.pt')

# Upload
api.upload_file(
    path_or_fileobj='/kaggle/working/chestai_best.pt',
    path_in_repo='chestai_best.pt',
    repo_id=HF_REPO,
)
api.upload_file(
    path_or_fileobj='configs/config.yaml',
    path_in_repo='config.yaml',
    repo_id=HF_REPO,
)
api.upload_file(
    path_or_fileobj='fairness_report.json',
    path_in_repo='fairness_report.json',
    repo_id=HF_REPO,
)

print(f'Model uploaded to: https://huggingface.co/{HF_REPO}')
print(f'Best AUC: {trainer.best_auc:.4f}')